# ARRGO: Testing and Validation

This notebook validates the correctness, consistency, and numerical robustness of the ARRGO implementation developed in the previous notebooks.

The purpose of this notebook is not to introduce new optimization logic. Instead, it verifies that the implemented components behave consistently with the problem definition, theoretical foundations, and algorithm specification established in Notebooks 01–03.

The validation process is organized around the following objectives:

1. Validate the core data structures and invariants.
2. Validate objective evaluation and evaluation-budget handling.
3. Validate evaluation history and duplicate-point prevention.
4. Validate region geometry and region hierarchy.
5. Validate candidate generation and feasibility.
6. Validate sampling and structural refinement behavior.
7. Validate numerical robustness and tolerance handling.
8. Validate global incumbent monotonicity.
9. Validate certified bounds when Certified Mode is enabled.
10. Validate deterministic execution of the complete ARRGO pipeline.

The tests distinguish between implementation correctness and optimization performance.

A test passing means that an implementation component satisfies its specified computational contract. It does not by itself establish that ARRGO is globally optimal or superior to other optimization methods.

Optimization performance will be evaluated separately using benchmark experiments and comparative evaluation.

## Validation Philosophy

ARRGO is validated at three levels:

- **Component level:** individual functions and data structures are tested in isolation.
- **Invariant level:** global properties that must remain true throughout execution are checked.
- **End-to-end level:** the complete ARRGO pipeline is executed and its final state is validated.

The validation process follows the principle:

$$
\text{Specification}
\rightarrow
\text{Implementation}
\rightarrow
\text{Tests}
\rightarrow
\text{Validation Evidence}
$$

A successful validation therefore provides evidence that the implementation is consistent with the defined ARRGO algorithm and its stated assumptions.

## Validation Architecture

The validation process is divided into three complementary levels.

### 1. Component Validation

Component-level tests verify the local contracts of individual implementation units.

These tests cover:

- configuration and numerical tolerances
- objective evaluation
- evaluation history
- region representation
- region hierarchy
- region evaluation views
- region analysis
- candidate generation
- candidate evaluation
- refinement actions
- certified bounds
- termination criteria

A component test should verify both valid behavior and appropriate rejection of invalid input.

### 2. Invariant Validation

Invariant-level tests verify properties that must remain valid throughout ARRGO execution.

The main invariants are:

- evaluation identifiers remain valid and unique
- evaluation points remain numerically distinct
- evaluation values remain finite
- the evaluation budget is never exceeded
- region boundaries remain valid
- child regions remain inside their parent regions
- binary children form a continuous partition of their parent
- refinement satisfies the contraction condition
- parent regions remain represented after refinement
- previously collected evaluations are preserved
- the global incumbent is monotone non-decreasing
- deterministic execution produces consistent results

These properties are independent of the particular objective function whenever possible.

### 3. End-to-End Validation

End-to-end tests execute the complete ARRGO pipeline on controlled objective functions.

The end-to-end validation verifies:

- initialization
- initial sampling
- iterative refinement
- global region selection
- sampling and splitting
- incumbent updates
- budget handling
- termination
- final result construction

Both empirical and certified execution modes are validated separately.

## Validation Categories

The tests are organized into the following categories:

1. Configuration Validation
2. Objective Evaluation Validation
3. Evaluation History Validation
4. Region and Hierarchy Validation
5. Candidate Generation Validation
6. Refinement Validation
7. Numerical Robustness Validation
8. Global State Validation
9. Certified Bound Validation
10. Termination Validation
11. Determinism Validation
12. End-to-End Validation

The tests are designed to provide evidence for implementation correctness without conflating correctness with optimization performance.

Performance, convergence behavior, benchmark results, and comparisons with other methods are evaluated in later notebooks.

In [1]:
def assert_close(actual: float, expected: float, tolerance: float = 1e-10) -> None:
    assert abs(actual - expected) <= tolerance


def assert_raises(exception_type: type[BaseException], function, *args, **kwargs) -> None:
    try:
        function(*args, **kwargs)
    except exception_type:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {exception_type.__name__}, but got {type(exc).__name__}."
        ) from exc

    raise AssertionError(
        f"Expected {exception_type.__name__} to be raised."
    )


def assert_finite(value: float) -> None:
    assert value == value
    assert abs(value) != float("inf")


def assert_sequence_equal(actual, expected) -> None:
    assert tuple(actual) == tuple(expected)


print("Test utilities loaded.")

Test utilities loaded.


In [2]:
from arrgo import ARRGOConfig, optimize


def test_smoke_run() -> None:
    def objective(x: float) -> float:
        return -(x - 2.0) ** 2 + 5.0

    config = ARRGOConfig(
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=20,
    )

    result = optimize(
        objective=objective,
        config=config,
    )

    assert result.best_x is not None
    assert result.best_value is not None
    assert result.evaluation_count > 0
    assert result.evaluation_count <= config.max_evaluations


test_smoke_run()

print("Smoke test passed.")

Smoke test passed.


In [3]:
from arrgo import ARRGOConfig, ARRGOExecutionMode


def test_invalid_domain() -> None:
    assert_raises(
        ValueError,
        ARRGOConfig,
        lower_bound=5.0,
        upper_bound=-5.0,
        max_evaluations=20,
    )


def test_invalid_evaluation_budget() -> None:
    assert_raises(
        ValueError,
        ARRGOConfig,
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=0,
    )


def test_invalid_contraction_factor() -> None:
    assert_raises(
        ValueError,
        ARRGOConfig,
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=20,
        contraction_factor=1.0,
    )


def test_certified_mode_requires_lipschitz_constant() -> None:
    assert_raises(
        ValueError,
        ARRGOConfig,
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=20,
        mode=ARRGOExecutionMode.CERTIFIED,
    )


def test_certified_mode_requires_epsilon() -> None:
    assert_raises(
        ValueError,
        ARRGOConfig,
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=20,
        mode=ARRGOExecutionMode.CERTIFIED,
        lipschitz_constant=2.0,
    )


def test_valid_certified_configuration() -> None:
    config = ARRGOConfig(
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=20,
        mode=ARRGOExecutionMode.CERTIFIED,
        lipschitz_constant=2.0,
        epsilon=0.01,
    )

    assert config.mode == ARRGOExecutionMode.CERTIFIED
    assert config.lipschitz_constant == 2.0
    assert config.epsilon == 0.01


test_invalid_domain()
test_invalid_evaluation_budget()
test_invalid_contraction_factor()
test_certified_mode_requires_lipschitz_constant()
test_certified_mode_requires_epsilon()
test_valid_certified_configuration()

print("Configuration validation tests passed.")

Configuration validation tests passed.


In [4]:
from arrgo import Evaluation, EvaluationHistory


def test_evaluation_history_starts_empty() -> None:
    history = EvaluationHistory()

    assert history.count == 0
    assert history.evaluations == ()
    assert history.best() is None


def test_evaluation_history_add() -> None:
    history = EvaluationHistory()

    evaluation = Evaluation(
        x=2.0,
        value=5.0,
        id=0,
    )

    history.add(evaluation)

    assert history.count == 1
    assert history.evaluations == (evaluation,)


def test_evaluation_history_best() -> None:
    history = EvaluationHistory()

    evaluations = (
        Evaluation(x=-1.0, value=2.0, id=0),
        Evaluation(x=0.0, value=5.0, id=1),
        Evaluation(x=1.0, value=3.0, id=2),
    )

    history.extend(evaluations)

    best = history.best()

    assert best is not None
    assert best.x == 0.0
    assert best.value == 5.0
    assert best.id == 1


def test_evaluation_history_contains_x() -> None:
    history = EvaluationHistory()

    history.add(
        Evaluation(
            x=2.0,
            value=5.0,
            id=0,
        )
    )

    assert history.contains_x(
        x=2.0,
        tolerance=1e-12,
    )

    assert history.contains_x(
        x=2.0 + 1e-13,
        tolerance=1e-12,
    )

    assert not history.contains_x(
        x=2.1,
        tolerance=1e-12,
    )


def test_evaluation_history_get_by_x() -> None:
    history = EvaluationHistory()

    evaluation = Evaluation(
        x=2.0,
        value=5.0,
        id=0,
    )

    history.add(evaluation)

    found = history.get_by_x(
        x=2.0 + 1e-13,
        tolerance=1e-12,
    )

    assert found == evaluation

    missing = history.get_by_x(
        x=3.0,
        tolerance=1e-12,
    )

    assert missing is None


test_evaluation_history_starts_empty()
test_evaluation_history_add()
test_evaluation_history_best()
test_evaluation_history_contains_x()
test_evaluation_history_get_by_x()

print("Evaluation history validation tests passed.")

Evaluation history validation tests passed.


In [5]:
from arrgo import Region, RegionHierarchy, RegionState


def test_region_geometry() -> None:
    region = Region(
        id=0,
        left_x=-5.0,
        right_x=5.0,
    )

    assert region.width == 10.0
    assert region.midpoint == 0.0


def test_region_contains() -> None:
    region = Region(
        id=0,
        left_x=-5.0,
        right_x=5.0,
    )

    assert region.contains(0.0)
    assert region.contains(-5.0)
    assert region.contains(5.0)

    assert not region.contains(6.0)


def test_region_contains_with_tolerance() -> None:
    region = Region(
        id=0,
        left_x=-5.0,
        right_x=5.0,
    )

    assert region.contains(
        5.0 + 1e-13,
        tolerance=1e-12,
    )

    assert not region.contains(
        5.0 + 1e-6,
        tolerance=1e-12,
    )


def test_region_state() -> None:
    region = Region(
        id=0,
        left_x=-5.0,
        right_x=5.0,
    )

    assert region.state == RegionState.ACTIVE

    region.state = RegionState.STABLE

    assert region.state == RegionState.STABLE


def test_region_hierarchy() -> None:
    hierarchy = RegionHierarchy()

    root = Region(
        id=0,
        left_x=-5.0,
        right_x=5.0,
    )

    left_child = Region(
        id=1,
        left_x=-5.0,
        right_x=0.0,
        parent_id=0,
    )

    right_child = Region(
        id=2,
        left_x=0.0,
        right_x=5.0,
        parent_id=0,
    )

    hierarchy.add_region(root)
    hierarchy.add_region(left_child)
    hierarchy.add_region(right_child)

    hierarchy.add_child_relationship(
        parent_id=0,
        child_id=1,
    )

    hierarchy.add_child_relationship(
        parent_id=0,
        child_id=2,
    )

    assert hierarchy.contains(0)
    assert hierarchy.contains(1)
    assert hierarchy.contains(2)

    assert hierarchy.get_region(0) == root

    assert tuple(root.children_ids) == (1, 2)

    assert left_child.parent_id == 0
    assert right_child.parent_id == 0

    roots = hierarchy.roots()

    assert roots == (root,)


def test_duplicate_region_id_is_rejected() -> None:
    hierarchy = RegionHierarchy()

    hierarchy.add_region(
        Region(
            id=0,
            left_x=-5.0,
            right_x=5.0,
        )
    )

    assert_raises(
        ValueError,
        hierarchy.add_region,
        Region(
            id=0,
            left_x=-2.0,
            right_x=2.0,
        ),
    )


test_region_geometry()
test_region_contains()
test_region_contains_with_tolerance()
test_region_state()
test_region_hierarchy()
test_duplicate_region_id_is_rejected()

print("Region and hierarchy validation tests passed.")

Region and hierarchy validation tests passed.


In [6]:
from arrgo import Evaluation, Region, analyze_region


def test_region_analysis() -> None:
    region = Region(
        id=0,
        left_x=0.0,
        right_x=10.0,
    )

    evaluations = (
        Evaluation(x=0.0, value=0.0, id=0),
        Evaluation(x=5.0, value=5.0, id=1),
        Evaluation(x=10.0, value=0.0, id=2),
    )

    analysis = analyze_region(
        region=region,
        evaluations=evaluations,
    )

    assert analysis.region_id == 0
    assert analysis.evaluation_count == 3

    assert analysis.best_value == 5.0
    assert analysis.worst_value == 0.0
    assert analysis.value_range == 5.0

    assert analysis.coverage_resolution == 5.0

    assert analysis.behavior_resolution == 2.0

    assert analysis.observed_uncertainty == 5.0
    assert analysis.observed_potential == 5.0


def test_empty_region_analysis() -> None:
    region = Region(
        id=1,
        left_x=-5.0,
        right_x=5.0,
    )

    analysis = analyze_region(
        region=region,
        evaluations=(),
    )

    assert analysis.region_id == 1
    assert analysis.evaluation_count == 0

    assert analysis.best_value is None
    assert analysis.worst_value is None
    assert analysis.value_range is None
    assert analysis.coverage_resolution is None
    assert analysis.behavior_resolution is None
    assert analysis.observed_uncertainty is None
    assert analysis.observed_potential is None


def test_region_analysis_ignores_outside_evaluations() -> None:
    region = Region(
        id=2,
        left_x=0.0,
        right_x=10.0,
    )

    evaluations = (
        Evaluation(x=-5.0, value=100.0, id=0),
        Evaluation(x=0.0, value=1.0, id=1),
        Evaluation(x=10.0, value=3.0, id=2),
        Evaluation(x=15.0, value=200.0, id=3),
    )

    analysis = analyze_region(
        region=region,
        evaluations=evaluations,
    )

    assert analysis.evaluation_count == 2
    assert analysis.best_value == 3.0
    assert analysis.worst_value == 1.0
    assert analysis.value_range == 2.0
    assert analysis.coverage_resolution == 10.0
    assert analysis.observed_uncertainty == 2.0
    assert analysis.observed_potential == 3.0


test_region_analysis()
test_empty_region_analysis()
test_region_analysis_ignores_outside_evaluations()

print("Region analysis validation tests passed.")

Region analysis validation tests passed.


In [7]:
from arrgo import (
    Candidate,
    Evaluation,
    RefinementAction,
    Region,
    generate_sampling_candidates,
    select_sampling_candidate,
)


def test_generate_sampling_candidates_without_evaluations() -> None:
    region = Region(
        id=0,
        left_x=-5.0,
        right_x=5.0,
    )

    candidates = generate_sampling_candidates(
        region=region,
        evaluations=(),
    )

    assert len(candidates) == 1

    candidate = candidates[0]

    assert candidate.action == RefinementAction.SAMPLE
    assert candidate.region_id == 0
    assert candidate.location == 0.0


def test_generate_sampling_candidates() -> None:
    region = Region(
        id=1,
        left_x=0.0,
        right_x=10.0,
    )

    evaluations = (
        Evaluation(x=0.0, value=0.0, id=0),
        Evaluation(x=5.0, value=5.0, id=1),
        Evaluation(x=10.0, value=0.0, id=2),
    )

    candidates = generate_sampling_candidates(
        region=region,
        evaluations=evaluations,
    )

    locations = tuple(
        candidate.location
        for candidate in candidates
    )

    assert 0.0 not in locations
    assert 5.0 not in locations
    assert 10.0 not in locations

    assert 2.5 in locations
    assert 7.5 in locations

    assert all(
        region.contains(candidate.location)
        for candidate in candidates
    )

    assert all(
        candidate.action == RefinementAction.SAMPLE
        for candidate in candidates
    )


def test_sampling_candidates_are_unique() -> None:
    region = Region(
        id=2,
        left_x=0.0,
        right_x=10.0,
    )

    evaluations = (
        Evaluation(x=0.0, value=0.0, id=0),
        Evaluation(x=5.0, value=5.0, id=1),
        Evaluation(x=10.0, value=0.0, id=2),
    )

    candidates = generate_sampling_candidates(
        region=region,
        evaluations=evaluations,
    )

    locations = [
        candidate.location
        for candidate in candidates
    ]

    assert len(locations) == len(set(locations))


def test_select_sampling_candidate() -> None:
    region = Region(
        id=3,
        left_x=0.0,
        right_x=10.0,
    )

    evaluations = (
        Evaluation(x=0.0, value=0.0, id=0),
        Evaluation(x=5.0, value=5.0, id=1),
        Evaluation(x=10.0, value=0.0, id=2),
    )

    candidates = generate_sampling_candidates(
        region=region,
        evaluations=evaluations,
    )

    selected = select_sampling_candidate(
        candidates=candidates,
        evaluations=evaluations,
    )

    assert selected is not None
    assert selected.action == RefinementAction.SAMPLE
    assert selected.region_id == 3

    assert selected.location in (
        2.5,
        7.5,
    )


def test_select_sampling_candidate_returns_none_when_no_candidates() -> None:
    selected = select_sampling_candidate(
        candidates=(),
        evaluations=(),
    )

    assert selected is None


def test_sampling_candidate_representation() -> None:
    candidate = Candidate(
        location=2.5,
        action=RefinementAction.SAMPLE,
        region_id=4,
    )

    assert candidate.location == 2.5
    assert candidate.action == RefinementAction.SAMPLE
    assert candidate.region_id == 4
    assert candidate.score is None


test_generate_sampling_candidates_without_evaluations()
test_generate_sampling_candidates()
test_sampling_candidates_are_unique()
test_select_sampling_candidate()
test_select_sampling_candidate_returns_none_when_no_candidates()
test_sampling_candidate_representation()

print("Sampling candidate validation tests passed.")

Sampling candidate validation tests passed.


In [8]:
from arrgo import (
    Region,
    SplitCandidate,
    build_split_candidates,
    generate_split_locations,
)


def test_generate_split_locations_midpoint() -> None:
    region = Region(
        id=0,
        left_x=-5.0,
        right_x=5.0,
    )

    locations = generate_split_locations(
        region=region,
        contraction_factor=0.5,
    )

    assert locations == (0.0,)


def test_generate_split_locations_with_larger_contraction_factor() -> None:
    region = Region(
        id=1,
        left_x=0.0,
        right_x=10.0,
    )

    locations = generate_split_locations(
        region=region,
        contraction_factor=0.75,
    )

    assert locations == ( 
    2.5,
    5.0,
    7.5,
    )


def test_invalid_contraction_factor_returns_no_split_locations() -> None:
    region = Region(
        id=2,
        left_x=0.0,
        right_x=10.0,
    )

    locations = generate_split_locations(
        region=region,
        contraction_factor=0.25,
    )

    assert locations == ()


def test_split_locations_are_inside_region() -> None:
    region = Region(
        id=3,
        left_x=-5.0,
        right_x=5.0,
    )

    locations = generate_split_locations(
        region=region,
        contraction_factor=0.75,
    )

    assert all(
        region.left_x < location < region.right_x
        for location in locations
    )


def test_build_split_candidates() -> None:
    region = Region(
        id=4,
        left_x=-5.0,
        right_x=5.0,
    )

    candidates = build_split_candidates(
        region=region,
        contraction_factor=0.5,
    )

    assert len(candidates) == 1

    candidate = candidates[0]

    assert isinstance(candidate, SplitCandidate)
    assert candidate.region_id == 4
    assert candidate.split_location == 0.0
    assert candidate.contraction_valid is True


def test_split_candidates_are_valid() -> None:
    region = Region(
        id=5,
        left_x=0.0,
        right_x=10.0,
    )

    candidates = build_split_candidates(
        region=region,
        contraction_factor=0.75,
    )

    assert all(
        candidate.contraction_valid
        for candidate in candidates
    )

    assert all(
        region.left_x < candidate.split_location < region.right_x
        for candidate in candidates
    )


def test_no_split_candidates_when_contraction_is_too_strict() -> None:
    region = Region(
        id=6,
        left_x=0.0,
        right_x=10.0,
    )

    candidates = build_split_candidates(
        region=region,
        contraction_factor=0.25,
    )

    assert candidates == ()


test_generate_split_locations_midpoint()
test_generate_split_locations_with_larger_contraction_factor()
test_invalid_contraction_factor_returns_no_split_locations()
test_split_locations_are_inside_region()
test_build_split_candidates()
test_split_candidates_are_valid()
test_no_split_candidates_when_contraction_is_too_strict()

print("Split candidate validation tests passed.")

Split candidate validation tests passed.


In [9]:
from arrgo import (
    Evaluation,
    Region,
    analyze_region,
    build_split_profile,
    calculate_coverage_difference,
    calculate_directional_difference,
    calculate_potential_difference,
    calculate_slope_variation_difference,
    calculate_structural_difference,
    calculate_uncertainty_difference,
)


def test_structural_difference() -> None:
    region = Region(
        id=0,
        left_x=0.0,
        right_x=10.0,
    )

    assert calculate_structural_difference(
        region=region,
        split_location=5.0,
    ) == 0.0

    assert calculate_structural_difference(
        region=region,
        split_location=3.0,
    ) == 4.0


def test_directional_difference() -> None:
    region = Region(
        id=1,
        left_x=0.0,
        right_x=10.0,
    )

    evaluations = (
        Evaluation(x=0.0, value=0.0, id=0),
        Evaluation(x=2.0, value=0.0, id=1),
        Evaluation(x=8.0, value=10.0, id=2),
        Evaluation(x=10.0, value=10.0, id=3),
    )

    difference = calculate_directional_difference(
        region=region,
        evaluations=evaluations,
        split_location=5.0,
    )

    assert difference == 10.0


def test_slope_variation_difference() -> None:
    region = Region(
        id=2,
        left_x=0.0,
        right_x=10.0,
    )

    evaluations = (
        Evaluation(x=0.0, value=0.0, id=0),
        Evaluation(x=2.0, value=2.0, id=1),
        Evaluation(x=5.0, value=5.0, id=2),
        Evaluation(x=7.0, value=9.0, id=3),
        Evaluation(x=10.0, value=15.0, id=4),
    )

    difference = calculate_slope_variation_difference(
        region=region,
        evaluations=evaluations,
        split_location=5.0,
    )

    assert difference == 0.0


def test_coverage_difference() -> None:
    region = Region(
        id=3,
        left_x=0.0,
        right_x=10.0,
    )

    evaluations = (
        Evaluation(x=0.0, value=0.0, id=0),
        Evaluation(x=2.0, value=2.0, id=1),
        Evaluation(x=5.0, value=5.0, id=2),
        Evaluation(x=10.0, value=10.0, id=3),
    )

    difference = calculate_coverage_difference(
        region=region,
        evaluations=evaluations,
        split_location=5.0,
    )

    assert difference == 2.0


def test_uncertainty_difference() -> None:
    assert calculate_uncertainty_difference(
        left_uncertainty=2.0,
        right_uncertainty=5.0,
    ) == 3.0

    assert calculate_uncertainty_difference(
        left_uncertainty=None,
        right_uncertainty=5.0,
    ) == 0.0


def test_potential_difference() -> None:
    assert calculate_potential_difference(
        left_potential=2.0,
        right_potential=7.0,
    ) == 5.0

    assert calculate_potential_difference(
        left_potential=None,
        right_potential=7.0,
    ) == 0.0


def test_build_split_profile() -> None:
    region = Region(
        id=4,
        left_x=0.0,
        right_x=10.0,
    )

    evaluations = (
        Evaluation(x=0.0, value=0.0, id=0),
        Evaluation(x=2.0, value=0.0, id=1),
        Evaluation(x=5.0, value=5.0, id=2),
        Evaluation(x=8.0, value=10.0, id=3),
        Evaluation(x=10.0, value=10.0, id=4),
    )

    split_location = 5.0

    left_region = Region(
        id=5,
        left_x=0.0,
        right_x=5.0,
    )

    right_region = Region(
        id=6,
        left_x=5.0,
        right_x=10.0,
    )

    left_analysis = analyze_region(
        region=left_region,
        evaluations=evaluations,
    )

    right_analysis = analyze_region(
        region=right_region,
        evaluations=evaluations,
    )

    profile = build_split_profile(
        region=region,
        evaluations=evaluations,
        split_location=split_location,
        left_analysis=left_analysis,
        right_analysis=right_analysis,
    )

    assert profile.structural == 0.0
    assert profile.directional >= 0.0
    assert profile.slope_variation >= 0.0
    assert profile.coverage >= 0.0
    assert profile.uncertainty >= 0.0
    assert profile.potential >= 0.0


test_structural_difference()
test_directional_difference()
test_slope_variation_difference()
test_coverage_difference()
test_uncertainty_difference()
test_potential_difference()
test_build_split_profile()

print("Split profile validation tests passed.")

Split profile validation tests passed.


In [10]:
from arrgo import (
    SplitCandidate,
    StructuralDifferenceProfile,
    dominates,
    select_non_dominated_splits,
)


def test_dominance_requires_all_dimensions() -> None:
    first = StructuralDifferenceProfile(
        structural=5.0,
        directional=5.0,
        slope_variation=5.0,
        coverage=5.0,
        uncertainty=5.0,
        potential=5.0,
    )

    second = StructuralDifferenceProfile(
        structural=4.0,
        directional=4.0,
        slope_variation=4.0,
        coverage=4.0,
        uncertainty=4.0,
        potential=4.0,
    )

    assert dominates(first, second)
    assert not dominates(second, first)


def test_equal_profiles_do_not_dominate_each_other() -> None:
    profile = StructuralDifferenceProfile(
        structural=1.0,
        directional=2.0,
        slope_variation=3.0,
        coverage=4.0,
        uncertainty=5.0,
        potential=6.0,
    )

    same_profile = StructuralDifferenceProfile(
        structural=1.0,
        directional=2.0,
        slope_variation=3.0,
        coverage=4.0,
        uncertainty=5.0,
        potential=6.0,
    )

    assert not dominates(profile, same_profile)
    assert not dominates(same_profile, profile)


def test_tradeoff_profiles_are_non_dominating() -> None:
    first = StructuralDifferenceProfile(
        structural=10.0,
        directional=1.0,
        slope_variation=5.0,
        coverage=5.0,
        uncertainty=5.0,
        potential=5.0,
    )

    second = StructuralDifferenceProfile(
        structural=1.0,
        directional=10.0,
        slope_variation=5.0,
        coverage=5.0,
        uncertainty=5.0,
        potential=5.0,
    )

    assert not dominates(first, second)
    assert not dominates(second, first)


def test_select_non_dominated_splits() -> None:
    candidates = (
        SplitCandidate(
            region_id=0,
            split_location=2.0,
            contraction_valid=True,
        ),
        SplitCandidate(
            region_id=0,
            split_location=5.0,
            contraction_valid=True,
        ),
        SplitCandidate(
            region_id=0,
            split_location=8.0,
            contraction_valid=True,
        ),
    )

    profiles = (
        StructuralDifferenceProfile(
            structural=5.0,
            directional=5.0,
            slope_variation=5.0,
            coverage=5.0,
            uncertainty=5.0,
            potential=5.0,
        ),
        StructuralDifferenceProfile(
            structural=3.0,
            directional=3.0,
            slope_variation=3.0,
            coverage=3.0,
            uncertainty=3.0,
            potential=3.0,
        ),
        StructuralDifferenceProfile(
            structural=5.0,
            directional=2.0,
            slope_variation=6.0,
            coverage=2.0,
            uncertainty=6.0,
            potential=4.0,
        ),
    )

    selected = select_non_dominated_splits(
        candidates=candidates,
        profiles=profiles,
    )

    assert candidates[1] not in selected
    assert candidates[0] in selected
    assert candidates[2] in selected


def test_select_non_dominated_splits_requires_matching_lengths() -> None:
    candidates = (
        SplitCandidate(
            region_id=0,
            split_location=5.0,
            contraction_valid=True,
        ),
    )

    profiles = ()

    try:
        select_non_dominated_splits(
            candidates=candidates,
            profiles=profiles,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Expected ValueError for mismatched candidates and profiles."
        )


test_dominance_requires_all_dimensions()
test_equal_profiles_do_not_dominate_each_other()
test_tradeoff_profiles_are_non_dominating()
test_select_non_dominated_splits()
test_select_non_dominated_splits_requires_matching_lengths()

print("Dominance and non-dominated split validation tests passed.")

Dominance and non-dominated split validation tests passed.


In [11]:
from arrgo import (
    Candidate,
    RefinementAction,
    Region,
    RegionAnalysis,
    ResolutionCriteria,
    SplitCandidate,
    select_refinement_action,
)


def test_refinement_action_selects_sample_when_sampling_is_available() -> None:
    analysis = RegionAnalysis(
        region_id=0,
        evaluation_count=2,
        best_value=5.0,
        worst_value=1.0,
        value_range=4.0,
        coverage_resolution=2.0,
        behavior_resolution=1.0,
        observed_uncertainty=4.0,
        observed_potential=5.0,
    )

    sampling_candidates = (
        Candidate(
            location=2.5,
            action=RefinementAction.SAMPLE,
            region_id=0,
        ),
    )

    action = select_refinement_action(
        analysis=analysis,
        sampling_candidates=sampling_candidates,
        split_candidate=None,
    )

    assert action == RefinementAction.SAMPLE


def test_refinement_action_selects_split_when_split_is_the_available_action() -> None:
    analysis = RegionAnalysis(
        region_id=1,
        evaluation_count=2,
        best_value=5.0,
        worst_value=1.0,
        value_range=4.0,
        coverage_resolution=0.0,
        behavior_resolution=5.0,
        observed_uncertainty=4.0,
        observed_potential=5.0,
    )

    split_candidate = SplitCandidate(
        region_id=1,
        split_location=5.0,
        contraction_valid=True,
    )

    action = select_refinement_action(
        analysis=analysis,
        sampling_candidates=(),
        split_candidate=split_candidate,
    )

    assert action == RefinementAction.SPLIT


def test_refinement_action_selects_stable_when_information_is_resolved() -> None:
    analysis = RegionAnalysis(
        region_id=2,
        evaluation_count=3,
        best_value=5.0,
        worst_value=5.0,
        value_range=0.0,
        coverage_resolution=0.0,
        behavior_resolution=0.0,
        observed_uncertainty=0.0,
        observed_potential=0.0,
    )

    action = select_refinement_action(
        analysis=analysis,
        sampling_candidates=(),
        split_candidate=None,
        criteria=ResolutionCriteria(),
    )

    assert action == RefinementAction.STABLE


def test_refinement_action_falls_back_to_split_when_no_sampling_candidate_exists() -> None:
    analysis = RegionAnalysis(
        region_id=3,
        evaluation_count=2,
        best_value=5.0,
        worst_value=1.0,
        value_range=4.0,
        coverage_resolution=2.0,
        behavior_resolution=1.0,
        observed_uncertainty=4.0,
        observed_potential=5.0,
    )

    split_candidate = SplitCandidate(
        region_id=3,
        split_location=5.0,
        contraction_valid=True,
    )

    action = select_refinement_action(
        analysis=analysis,
        sampling_candidates=(),
        split_candidate=split_candidate,
    )

    assert action == RefinementAction.SPLIT


def test_refinement_action_is_stable_when_no_action_is_available() -> None:
    analysis = RegionAnalysis(
        region_id=4,
        evaluation_count=1,
        best_value=5.0,
        worst_value=5.0,
        value_range=0.0,
        coverage_resolution=5.0,
        behavior_resolution=0.0,
        observed_uncertainty=0.0,
        observed_potential=5.0,
    )

    action = select_refinement_action(
        analysis=analysis,
        sampling_candidates=(),
        split_candidate=None,
    )

    assert action == RefinementAction.STABLE


test_refinement_action_selects_sample_when_sampling_is_available()
test_refinement_action_selects_split_when_split_is_the_available_action()
test_refinement_action_selects_stable_when_information_is_resolved()
test_refinement_action_falls_back_to_split_when_no_sampling_candidate_exists()
test_refinement_action_is_stable_when_no_action_is_available()

print("Refinement decision validation tests passed.")

Refinement decision validation tests passed.


In [12]:
from arrgo import (
    Evaluation,
    Region,
    RegionState,
    select_global_region,
)


def test_global_region_selection_ignores_non_active_regions() -> None:
    regions = (
        Region(
            id=0,
            left_x=-5.0,
            right_x=0.0,
            state=RegionState.REFINED,
        ),
        Region(
            id=1,
            left_x=0.0,
            right_x=5.0,
            state=RegionState.ACTIVE,
        ),
    )

    evaluations = (
        Evaluation(x=-2.5, value=100.0, id=0),
        Evaluation(x=2.5, value=5.0, id=1),
    )

    selected = select_global_region(
        regions=regions,
        evaluations=evaluations,
    )

    assert selected is not None
    assert selected.id == 1


def test_global_region_selection_returns_none_without_active_regions() -> None:
    regions = (
        Region(
            id=0,
            left_x=-5.0,
            right_x=0.0,
            state=RegionState.REFINED,
        ),
        Region(
            id=1,
            left_x=0.0,
            right_x=5.0,
            state=RegionState.STABLE,
        ),
    )

    evaluations = (
        Evaluation(x=-2.5, value=5.0, id=0),
        Evaluation(x=2.5, value=10.0, id=1),
    )

    selected = select_global_region(
        regions=regions,
        evaluations=evaluations,
    )

    assert selected is None


def test_global_region_selection_prefers_more_unresolved_information() -> None:
    regions = (
        Region(
            id=0,
            left_x=-5.0,
            right_x=0.0,
            state=RegionState.ACTIVE,
        ),
        Region(
            id=1,
            left_x=0.0,
            right_x=5.0,
            state=RegionState.ACTIVE,
        ),
    )

    evaluations = (
        Evaluation(x=-5.0, value=1.0, id=0),
        Evaluation(x=0.0, value=1.0, id=1),
        Evaluation(x=5.0, value=1.0, id=2),
    )

    selected = select_global_region(
        regions=regions,
        evaluations=evaluations,
    )

    assert selected is not None
    assert selected.id in (0, 1)


def test_global_region_selection_is_deterministic() -> None:
    regions = (
        Region(
            id=0,
            left_x=-5.0,
            right_x=0.0,
            state=RegionState.ACTIVE,
        ),
        Region(
            id=1,
            left_x=0.0,
            right_x=5.0,
            state=RegionState.ACTIVE,
        ),
    )

    evaluations = (
        Evaluation(x=-5.0, value=1.0, id=0),
        Evaluation(x=0.0, value=1.0, id=1),
        Evaluation(x=5.0, value=1.0, id=2),
    )

    first = select_global_region(
        regions=regions,
        evaluations=evaluations,
    )

    second = select_global_region(
        regions=regions,
        evaluations=evaluations,
    )

    assert first is not None
    assert second is not None
    assert first.id == second.id


test_global_region_selection_ignores_non_active_regions()
test_global_region_selection_returns_none_without_active_regions()
test_global_region_selection_prefers_more_unresolved_information()
test_global_region_selection_is_deterministic()

print("Global region selection validation tests passed.")

Global region selection validation tests passed.


In [13]:
from arrgo import (
    ARRGOConfig,
    ARRGOGlobalState,
    EvaluationHistory,
    Region,
    RegionHierarchy,
    RegionState,
    create_child_regions,
    split_region,
)


def test_create_child_regions() -> None:
    state = ARRGOGlobalState(
        evaluation_history=EvaluationHistory(),
        region_hierarchy=RegionHierarchy(),
    )

    parent = Region(
        id=0,
        left_x=0.0,
        right_x=10.0,
    )

    state.region_hierarchy.add_region(parent)

    left_child, right_child = create_child_regions(
        state=state,
        parent_region=parent,
        split_location=5.0,
    )

    assert left_child.left_x == 0.0
    assert left_child.right_x == 5.0

    assert right_child.left_x == 5.0
    assert right_child.right_x == 10.0

    assert left_child.parent_id == 0
    assert right_child.parent_id == 0

    assert left_child.state == RegionState.ACTIVE
    assert right_child.state == RegionState.ACTIVE

    assert parent.children_ids == [
        left_child.id,
        right_child.id,
    ]

    assert parent.state == RegionState.REFINED


def test_parent_region_remains_in_hierarchy_after_split() -> None:
    state = ARRGOGlobalState(
        evaluation_history=EvaluationHistory(),
        region_hierarchy=RegionHierarchy(),
    )

    parent = Region(
        id=0,
        left_x=-5.0,
        right_x=5.0,
    )

    state.region_hierarchy.add_region(parent)

    left_child, right_child = split_region(
        state=state,
        region=parent,
        split_location=0.0,
    )

    assert state.region_hierarchy.contains(0)
    assert state.region_hierarchy.contains(left_child.id)
    assert state.region_hierarchy.contains(right_child.id)

    assert state.region_hierarchy.get_region(0) == parent


def test_split_region_requires_active_region() -> None:
    state = ARRGOGlobalState(
        evaluation_history=EvaluationHistory(),
        region_hierarchy=RegionHierarchy(),
    )

    parent = Region(
        id=0,
        left_x=0.0,
        right_x=10.0,
        state=RegionState.STABLE,
    )

    state.region_hierarchy.add_region(parent)

    try:
        split_region(
            state=state,
            region=parent,
            split_location=5.0,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Expected ValueError when splitting a non-active region."
        )


def test_split_location_must_be_inside_region() -> None:
    state = ARRGOGlobalState(
        evaluation_history=EvaluationHistory(),
        region_hierarchy=RegionHierarchy(),
    )

    parent = Region(
        id=0,
        left_x=0.0,
        right_x=10.0,
    )

    state.region_hierarchy.add_region(parent)

    try:
        create_child_regions(
            state=state,
            parent_region=parent,
            split_location=0.0,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Expected ValueError for boundary split location."
        )


def test_region_cannot_be_split_twice() -> None:
    state = ARRGOGlobalState(
        evaluation_history=EvaluationHistory(),
        region_hierarchy=RegionHierarchy(),
    )

    parent = Region(
        id=0,
        left_x=0.0,
        right_x=10.0,
    )

    state.region_hierarchy.add_region(parent)

    create_child_regions(
        state=state,
        parent_region=parent,
        split_location=5.0,
    )

    try:
        create_child_regions(
            state=state,
            parent_region=parent,
            split_location=2.5,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Expected ValueError when splitting a region twice."
        )


test_create_child_regions()
test_parent_region_remains_in_hierarchy_after_split()
test_split_region_requires_active_region()
test_split_location_must_be_inside_region()
test_region_cannot_be_split_twice()

print("Region splitting and hierarchy validation tests passed.")

Region splitting and hierarchy validation tests passed.


In [14]:
from arrgo import (
    ARRGOGlobalState,
    EvaluationHistory,
    Region,
    RegionHierarchy,
    sample_region,
)


def test_sample_region_adds_evaluation() -> None:
    state = ARRGOGlobalState(
        evaluation_history=EvaluationHistory(),
        region_hierarchy=RegionHierarchy(),
    )

    region = Region(
        id=0,
        left_x=0.0,
        right_x=10.0,
    )

    state.region_hierarchy.add_region(region)

    def objective(x: float) -> float:
        return -(x - 5.0) ** 2 + 10.0

    evaluation = sample_region(
        state=state,
        objective=objective,
        location=5.0,
    )

    assert evaluation.x == 5.0
    assert evaluation.value == 10.0

    assert state.evaluation_history.count == 1


def test_sample_region_updates_existing_history() -> None:
    state = ARRGOGlobalState(
        evaluation_history=EvaluationHistory(),
        region_hierarchy=RegionHierarchy(),
    )

    region = Region(
        id=0,
        left_x=0.0,
        right_x=10.0,
    )

    state.region_hierarchy.add_region(region)

    def objective(x: float) -> float:
        return x * 2.0

    sample_region(
        state=state,
        objective=objective,
        location=2.0,
    )

    sample_region(
        state=state,
        objective=objective,
        location=8.0,
    )

    assert state.evaluation_history.count == 2

    values = [
        evaluation.value
        for evaluation in state.evaluation_history.evaluations
    ]

    assert values == [4.0, 16.0]



def test_sample_region_rejects_duplicate_evaluation() -> None:
    state = ARRGOGlobalState(
        evaluation_history=EvaluationHistory(),
        region_hierarchy=RegionHierarchy(),
    )

    region = Region(
        id=0,
        left_x=0.0,
        right_x=10.0,
    )

    state.region_hierarchy.add_region(region)

    def objective(x: float) -> float:
        return x

    sample_region(
        state=state,
        objective=objective,
        location=5.0,
    )

    assert_raises(
        ValueError,
        sample_region,
        state=state,
        objective=objective,
        location=5.0,
    )


test_sample_region_adds_evaluation()
test_sample_region_updates_existing_history()
test_sample_region_rejects_duplicate_evaluation()

print("Sampling execution validation tests passed.")

Sampling execution validation tests passed.


In [15]:
from arrgo import (
    ARRGOGlobalState,
    Evaluation,
    EvaluationHistory,
    RegionHierarchy,
    update_global_incumbent,
)


def test_global_incumbent_initially_empty() -> None:
    state = ARRGOGlobalState(
        evaluation_history=EvaluationHistory(),
        region_hierarchy=RegionHierarchy(),
    )

    update_global_incumbent(state)

    assert state.best_x is None
    assert state.best_value is None


def test_global_incumbent_selects_best_evaluation() -> None:
    state = ARRGOGlobalState(
        evaluation_history=EvaluationHistory(),
        region_hierarchy=RegionHierarchy(),
    )

    state.evaluation_history.add(
        Evaluation(id=0, x=0.0, value=1.0)
    )

    state.evaluation_history.add(
        Evaluation(id=1, x=2.0, value=5.0)
    )

    state.evaluation_history.add(
        Evaluation(id=2, x=4.0, value=3.0)
    )

    update_global_incumbent(state)

    assert state.best_x == 2.0
    assert state.best_value == 5.0


def test_global_incumbent_updates_when_better_evaluation_is_added() -> None:
    state = ARRGOGlobalState(
        evaluation_history=EvaluationHistory(),
        region_hierarchy=RegionHierarchy(),
    )

    state.evaluation_history.add(
        Evaluation(id=0, x=1.0, value=2.0)
    )

    update_global_incumbent(state)

    assert state.best_x == 1.0
    assert state.best_value == 2.0

    state.evaluation_history.add(
        Evaluation(id=1, x=3.0, value=8.0)
    )

    update_global_incumbent(state)

    assert state.best_x == 3.0
    assert state.best_value == 8.0


def test_global_incumbent_does_not_decrease() -> None:
    state = ARRGOGlobalState(
        evaluation_history=EvaluationHistory(),
        region_hierarchy=RegionHierarchy(),
    )

    state.evaluation_history.add(
        Evaluation(id=0, x=1.0, value=10.0)
    )

    update_global_incumbent(state)

    assert state.best_x == 1.0
    assert state.best_value == 10.0

    state.evaluation_history.add(
        Evaluation(id=1, x=2.0, value=4.0)
    )

    update_global_incumbent(state)

    assert state.best_x == 1.0
    assert state.best_value == 10.0


test_global_incumbent_initially_empty()
test_global_incumbent_selects_best_evaluation()
test_global_incumbent_updates_when_better_evaluation_is_added()
test_global_incumbent_does_not_decrease()

print("Global state and incumbent validation tests passed.")

Global state and incumbent validation tests passed.


In [16]:
from arrgo import (
    Evaluation,
    Region,
    calculate_exact_regional_bounds,
    calculate_lower_envelope,
    calculate_regional_potential,
    calculate_regional_uncertainty,
    calculate_upper_envelope,
)


def test_lower_envelope_is_below_upper_envelope() -> None:
    evaluations = (
        Evaluation(id=0, x=0.0, value=1.0),
        Evaluation(id=1, x=2.0, value=5.0),
    )

    lower = calculate_lower_envelope(
        x=1.0,
        evaluations=evaluations,
        lipschitz_constant=2.0,
    )

    upper = calculate_upper_envelope(
        x=1.0,
        evaluations=evaluations,
        lipschitz_constant=2.0,
    )

    assert lower <= upper


def test_regional_uncertainty_is_non_negative() -> None:
    evaluations = (
        Evaluation(id=0, x=0.0, value=1.0),
        Evaluation(id=1, x=2.0, value=5.0),
    )

    uncertainty = calculate_regional_uncertainty(
        x=1.0,
        evaluations=evaluations,
        lipschitz_constant=2.0,
    )

    assert uncertainty >= 0.0


def test_regional_potential_is_not_below_observed_values() -> None:
    evaluations = (
        Evaluation(id=0, x=0.0, value=1.0),
        Evaluation(id=1, x=2.0, value=5.0),
    )

    potential = calculate_regional_potential(
        x=2.0,
        evaluations=evaluations,
        lipschitz_constant=2.0,
    )

    assert potential >= 5.0


def test_exact_regional_bounds_are_finite() -> None:
    region = Region(
        id=0,
        left_x=0.0,
        right_x=2.0,
    )

    evaluations = (
        Evaluation(id=0, x=0.0, value=1.0),
        Evaluation(id=1, x=2.0, value=5.0),
    )

    bounds = calculate_exact_regional_bounds(
        region=region,
        evaluations=evaluations,
        lipschitz_constant=2.0,
    )

    assert_finite(bounds.lower_value)
    assert_finite(bounds.upper_value)
    assert_finite(bounds.uncertainty)
    assert_finite(bounds.potential)

    assert bounds.uncertainty >= 0.0
    assert bounds.potential >= bounds.lower_value


test_lower_envelope_is_below_upper_envelope()
test_regional_uncertainty_is_non_negative()
test_regional_potential_is_not_below_observed_values()
test_exact_regional_bounds_are_finite()

print("Certified bounds validation tests passed.")

Certified bounds validation tests passed.


In [19]:
from arrgo import (
    ARRGOConfig,
    ARRGOExecutionMode,
    TerminationReason,
    check_budget_termination,
    check_certified_termination,
    check_no_action_termination,
    check_numerical_termination,
)


def test_budget_termination() -> None:
    config = ARRGOConfig(
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=10,
    )

    status = check_budget_termination(
        evaluation_count=10,
        config=config,
    )

    assert status.should_terminate is True
    assert status.reason == TerminationReason.BUDGET_EXHAUSTED


def test_budget_does_not_terminate_before_limit() -> None:
    config = ARRGOConfig(
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=10,
    )

    status = check_budget_termination(
        evaluation_count=9,
        config=config,
    )

    assert status.should_terminate is False
    assert status.reason is None


def test_certified_termination() -> None:
    config = ARRGOConfig(
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=100,
        mode=ARRGOExecutionMode.CERTIFIED,
        lipschitz_constant=2.0,
        epsilon=0.1,
    )

    status = check_certified_termination(
        global_gap=0.05,
        config=config,
    )

    assert status.should_terminate is True
    assert status.reason == TerminationReason.CERTIFIED_TOLERANCE


def test_certified_mode_continues_when_gap_is_too_large() -> None:
    config = ARRGOConfig(
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=100,
        mode=ARRGOExecutionMode.CERTIFIED,
        lipschitz_constant=2.0,
        epsilon=0.1,
    )

    status = check_certified_termination(
        global_gap=0.5,
        config=config,
    )

    assert status.should_terminate is False
    assert status.reason is None


def test_empirical_mode_never_uses_certified_termination() -> None:
    config = ARRGOConfig(
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=100,
        mode=ARRGOExecutionMode.EMPIRICAL,
    )

    status = check_certified_termination(
        global_gap=0.0,
        config=config,
    )

    assert status.should_terminate is False
    assert status.reason is None


def test_numerical_termination() -> None:
    status = check_numerical_termination(
        numerical_failure=True,
    )

    assert status.should_terminate is True
    assert status.reason == TerminationReason.NUMERICAL_LIMIT


def test_numerical_failure_does_not_terminate_when_false() -> None:
    status = check_numerical_termination(
        numerical_failure=False,
    )

    assert status.should_terminate is False
    assert status.reason is None


def test_no_action_termination() -> None:
    status = check_no_action_termination(
        has_valid_action=False,
    )

    assert status.should_terminate is True
    assert status.reason == TerminationReason.NO_VALID_ACTION


def test_no_action_does_not_terminate_when_action_exists() -> None:
    status = check_no_action_termination(
        has_valid_action=True,
    )

    assert status.should_terminate is False
    assert status.reason is None


test_budget_termination()
test_budget_does_not_terminate_before_limit()
test_certified_termination()
test_certified_mode_continues_when_gap_is_too_large()
test_empirical_mode_never_uses_certified_termination()
test_numerical_termination()
test_numerical_failure_does_not_terminate_when_false()
test_no_action_termination()
test_no_action_does_not_terminate_when_action_exists()

print("Termination validation tests passed.")

Termination validation tests passed.


In [20]:
from arrgo import (
    ARRGOConfig,
    run_arrgo,
)


def test_arrgo_is_deterministic() -> None:
    def objective(x: float) -> float:
        return -(x - 2.0) ** 2 + 5.0

    config = ARRGOConfig(
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=30,
    )

    result_1 = run_arrgo(
        objective=objective,
        config=config,
    )

    result_2 = run_arrgo(
        objective=objective,
        config=config,
    )

    assert result_1.best_x == result_2.best_x
    assert result_1.best_value == result_2.best_value
    assert result_1.evaluation_count == result_2.evaluation_count
    assert result_1.termination_reason == result_2.termination_reason


test_arrgo_is_deterministic()

print("Determinism validation tests passed.")

Determinism validation tests passed.


In [21]:
from arrgo import (
    ARRGOConfig,
    ARRGOExecutionMode,
    TerminationReason,
    run_arrgo,
)


def test_end_to_end_empirical_run() -> None:
    def objective(x: float) -> float:
        return -(x - 2.0) ** 2 + 5.0

    config = ARRGOConfig(
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=30,
        mode=ARRGOExecutionMode.EMPIRICAL,
    )

    result = run_arrgo(
        objective=objective,
        config=config,
    )

    assert result.best_x is not None
    assert result.best_value is not None

    assert result.evaluation_count > 0
    assert result.evaluation_count <= config.max_evaluations

    assert -5.0 <= result.best_x <= 5.0

    assert result.best_value <= 5.0

    assert result.termination_reason in {
        TerminationReason.BUDGET_EXHAUSTED,
        TerminationReason.NO_VALID_ACTION,
        TerminationReason.NUMERICAL_LIMIT,
        TerminationReason.CERTIFIED_TOLERANCE,
    }


test_end_to_end_empirical_run()

print("End-to-end empirical validation tests passed.")

End-to-end empirical validation tests passed.


In [22]:
from arrgo import (
    ARRGOConfig,
    ARRGOExecutionMode,
    TerminationReason,
    run_arrgo,
)


def test_end_to_end_certified_run() -> None:
    def objective(x: float) -> float:
        return -(x - 2.0) ** 2 + 5.0

    config = ARRGOConfig(
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=50,
        mode=ARRGOExecutionMode.CERTIFIED,
        lipschitz_constant=10.0,
        epsilon=0.5,
    )

    result = run_arrgo(
        objective=objective,
        config=config,
    )

    assert result.best_x is not None
    assert result.best_value is not None

    assert result.evaluation_count > 0
    assert result.evaluation_count <= config.max_evaluations

    assert -5.0 <= result.best_x <= 5.0

    assert result.best_value <= 5.0

    assert result.termination_reason in {
        TerminationReason.CERTIFIED_TOLERANCE,
        TerminationReason.BUDGET_EXHAUSTED,
        TerminationReason.NO_VALID_ACTION,
        TerminationReason.NUMERICAL_LIMIT,
    }


test_end_to_end_certified_run()

print("End-to-end certified validation tests passed.")

End-to-end certified validation tests passed.


In [23]:
from arrgo import (
    ARRGOConfig,
    ARRGOExecutionMode,
    run_arrgo,
)


def test_numerical_robustness_with_close_points() -> None:
    def objective(x: float) -> float:
        return -(x - 1.0) ** 2

    config = ARRGOConfig(
        lower_bound=0.0,
        upper_bound=2.0,
        max_evaluations=30,
        coordinate_tolerance=1e-12,
        objective_tolerance=1e-12,
    )

    result = run_arrgo(
        objective=objective,
        config=config,
    )

    assert result.best_x is not None
    assert result.best_value is not None
    assert result.evaluation_count <= config.max_evaluations


def test_numerical_robustness_with_small_domain() -> None:
    def objective(x: float) -> float:
        return -(x - 1e-9) ** 2

    config = ARRGOConfig(
        lower_bound=0.0,
        upper_bound=2e-9,
        max_evaluations=20,
    )

    result = run_arrgo(
        objective=objective,
        config=config,
    )

    assert result.best_x is not None
    assert result.best_value is not None
    assert 0.0 <= result.best_x <= 2e-9


def test_numerical_robustness_with_large_objective_values() -> None:
    def objective(x: float) -> float:
        return 1e12 - (x - 2.0) ** 2

    config = ARRGOConfig(
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=20,
    )

    result = run_arrgo(
        objective=objective,
        config=config,
    )

    assert result.best_x is not None
    assert result.best_value is not None
    assert result.evaluation_count <= config.max_evaluations


def test_certified_mode_remains_numerically_stable() -> None:
    def objective(x: float) -> float:
        return -(x - 2.0) ** 2 + 5.0

    config = ARRGOConfig(
        lower_bound=-5.0,
        upper_bound=5.0,
        max_evaluations=40,
        mode=ARRGOExecutionMode.CERTIFIED,
        lipschitz_constant=10.0,
        epsilon=0.5,
    )

    result = run_arrgo(
        objective=objective,
        config=config,
    )

    assert result.best_x is not None
    assert result.best_value is not None
    assert result.evaluation_count <= config.max_evaluations


test_numerical_robustness_with_close_points()
test_numerical_robustness_with_small_domain()
test_numerical_robustness_with_large_objective_values()
test_certified_mode_remains_numerically_stable()

print("Numerical robustness validation tests passed.")

Numerical robustness validation tests passed.


# Final Validation Summary

Notebook 05 validated the ARRGO implementation across component-level, invariant-level, numerical, deterministic, and end-to-end execution scenarios.

## Validation Coverage

The validation suite covered:

1. Configuration validation
2. Evaluation history validation
3. Region and hierarchy validation
4. Region analysis validation
5. Sampling candidate validation
6. Split candidate validation
7. Split profile validation
8. Dominance and non-dominated split validation
9. Refinement decision validation
10. Global region selection validation
11. Region splitting and hierarchy update validation
12. Sampling execution validation
13. Global state and incumbent validation
14. Certified bounds validation
15. Termination validation
16. Determinism validation
17. End-to-end empirical execution
18. End-to-end certified execution
19. Numerical robustness validation

All implemented validation tests passed for the tested scenarios.

## Validated Properties

The validation process confirmed that the current implementation:

- preserves the configured domain constraints;
- maintains valid evaluation history and unique evaluation identifiers;
- maintains region and hierarchy consistency;
- generates feasible sampling and splitting candidates;
- respects contraction constraints for valid split candidates;
- performs deterministic refinement decisions;
- maintains the global incumbent consistently;
- preserves the monotonicity of the best observed objective value;
- computes finite certified-bound quantities for tested cases;
- applies termination conditions consistently;
- produces deterministic results for identical configurations and objectives;
- executes successfully in both Empirical and Certified modes;
- remains numerically stable for the tested sensitive cases.

## Certified Mode

Certified Mode was validated using a supplied Lipschitz constant and an epsilon tolerance.

The certification logic is interpreted under the assumption that the supplied Lipschitz constant is valid for the objective function over the configured domain.

Under this assumption, the implementation maintains the intended relationship:

$$
f_{\mathrm{best}}
\leq
f^\ast
\leq
P_{\mathrm{global}}
$$

and defines the global certification gap as:

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}.
$$

If the certified gap satisfies:

$$
\Delta_{\mathrm{global}}
\leq
\varepsilon,
$$

the incumbent is considered epsilon-optimal under the stated assumptions.

## Scope and Limitations

The validation suite establishes implementation consistency for the tested scenarios. It does not constitute a proof of global optimality for every possible objective function or configuration.

In particular:

- empirical mode provides a best-found solution rather than a mathematical optimality certificate;
- certified guarantees depend on the validity of the supplied Lipschitz constant;
- numerical validation covers representative sensitive cases rather than every possible floating-point failure mode;
- end-to-end tests validate execution and state consistency, not universal optimization performance;
- convergence guarantees depend on the theoretical assumptions established in the theoretical foundations.

Therefore, the validation results should be interpreted as evidence that the current implementation is consistent with the specified ARRGO design and its stated assumptions.